# Harmonized Landsat-Sentinel (HLS) Reflectance

### Cheyenne River Cottonwoods

In this notebook, we demonstrate how to access, visualize, and analyze HLS data directly from NASA Earthaccess. We place emphasis on the different modes of data access (cloud streaming and downloading locally) and demonstrate the application of CARE principles throughout the notebook. Our case study is focused on the riparian corridor of the Missouri River in Pine Ridge, SD, with a focus on cottonwood gallaries. HLS time-series, spectral indices, and textural features are used to characterize cottonwood along the river.

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import earthaccess
import holoviews as hv
import hvplot.xarray

from pathlib import Path

# This will ignore some warnings caused by holoviews
import warnings
warnings.simplefilter('ignore')

print(os.getcwd())
maindir = Path(os.getcwd()).parent

auth = earthaccess.login()
earthaccess.login(persist=True)

print("Ready to go !")

In [ ]:
# load the craven canyon study site
fp = os.path.join(maindir, 'data/CravenCanyon_StudyArea.gpkg')
aoi = gpd.read_file(fp)
aoi.plot()

In [ ]:
# make a bounding box
bounds = aoi.envelope
bounds.plot()

In [ ]:
from shapely.geometry import box
bbox_poly = box(*bounds.total_bounds)  # order is (minx, miny, maxx, maxy)
print(bbox_poly)
# Wrap in a GeoDataFrame
bbox_poly = gpd.GeoDataFrame(
    {"id": [1]},
    geometry=[bbox_poly],
    crs="EPSG:4326"
)

In [ ]:
print(tuple(bounds.total_bounds))

In [ ]:
# set up a search request for HLS data

first_date = "2025-06-01"
last_date = "2025-09-06"

granules = earthaccess.search_data(
    short_name=["HLSS30","HLSL30"],  # Sentinel-2 in HLS
    bounding_box=tuple(bounds.total_bounds),
    temporal=(first_date, last_date),
    cloud_hosted=True
)
print(f"Found {len(granules)} HLS Sentinel-2 granules")

In [ ]:
g = granules[0]
print(g["umm"].keys())
print(g["umm"]["AdditionalAttributes"][:5])

In [ ]:
def get_cloud_coverage(granule):
    """Extract CLOUD_COVERAGE from HLS Sentinel-2 granule metadata"""
    attrs = granule["umm"].get("AdditionalAttributes", [])
    for att in attrs:
        if att["Name"] == "CLOUD_COVERAGE":
            return float(att["Values"][0])
    return None

# Find the granule with the lowest cloud cover
best_granule = min(granules, key=lambda g: get_cloud_coverage(g) or 100)
print("Best granule:", best_granule["umm"]["GranuleUR"])
print("Cloud coverage:", get_cloud_coverage(best_granule), "%")
print("Date:", best_granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"])
pd.json_normalize(best_granule).columns

In [ ]:
# get URLs for reflectance file
urls = best_granule.data_links(access="external")
print(urls)

In [ ]:
# GDAL configurations used to successfully access LP DAAC Cloud Assets via vsicurl
from osgeo import gdal
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE','~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR', '~/cookies.txt')
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN','EMPTY_DIR')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS','TIF')
gdal.SetConfigOption('GDAL_HTTP_UNSAFESSL', 'YES')

In [ ]:
# subset bands to calculate EVI
evi_band_links = []

# Define which HLS product is being accessed
if urls[0].split('/')[4] == 'HLSS30.020':
    evi_bands = ['B8A', 'B04', 'B02', 'Fmask'] # NIR RED BLUE Quality for S30
else:
    evi_bands = ['B05', 'B04', 'B02', 'Fmask'] # NIR RED BLUE Quality for L30

# Subset the assets in the item down to only the desired bands
for a in urls:
    if any(b in a for b in evi_bands):
        evi_band_links.append(a)
evi_band_links

In [ ]:
# Use vsicurl to load the data directly into memory (be patient, may take a few seconds)

import rioxarray as rxr

chunk_size = dict(band=1, x=512, y=512) # Tiles have 1 band and are divided into 512x512 pixel chunks
bands = []
for e in evi_band_links:
    print(e)
    if e.rsplit('.', 2)[-2] == evi_bands[0]:      # NIR index
        nir = rxr.open_rasterio(e, chunks=chunk_size, masked=True).squeeze('band', drop=True)
        nir.attrs['scale_factor'] = 0.0001                         # hard coded the scale_factor attribute
    elif e.rsplit('.', 2)[-2] == evi_bands[1]:    # red index
        red = rxr.open_rasterio(e, chunks=chunk_size, masked=True).squeeze('band', drop=True)
        red.attrs['scale_factor'] = 0.0001                         # hard coded the scale_factor attribute
    elif e.rsplit('.', 2)[-2] == evi_bands[2]:    # blue index
        blue = rxr.open_rasterio(e, chunks=chunk_size, masked=True).squeeze('band', drop=True)
        blue.attrs['scale_factor'] = 0.0001                        # hard coded the scale_factor attribute
    # elif e.rsplit('.', 2)[-2] == evi_bands[3]:    # Fmask index
    #    fmask = rxr.open_rasterio(e, chunks=chunk_size, masked=True).astype('uint16').squeeze('band', drop=True)

print("The COGs have been loaded into memory!")

In [ ]:
nir

In [ ]:
# crop to our bounding box
# reproject the region to the raster CRS
bbox = bbox_poly.to_crs(nir.spatial_ref.crs_wkt) # Take the CRS from the NIR tile that we opened and apply it to our field geodataframe.
# crop each of the bands
nir = nir.rio.clip(bbox.geometry.values, bbox.crs, all_touched=True) # All touched includes any pixels touched by the polygon
red = red.rio.clip(bbox.geometry.values, bbox.crs, all_touched=True)
blue = blue.rio.clip(bbox.geometry.values, bbox.crs, all_touched=True)

# plot the NIR band
nir.hvplot.image(
    aspect='equal',
    cmap='magma',
    frame_height=500,
    frame_width= 500).opts(title='HLS Cropped NIR Band')  # Quick visual to assure that it worked

In [ ]:
# Define function to scale
def scaling(band):
    scale_factor = band.attrs['scale_factor']
    band_out = band.copy()
    band_out.data = band.data*scale_factor
    band_out.attrs['scale_factor'] = 1
    return(band_out)

# scale each band
nir_sc = scaling(nir)
red_sc = scaling(red)
blue_sc = scaling(blue)

In [ ]:
import xarray as xr
import numpy as np

def calc_evi(red, blue, nir):
    # Create EVI xarray.DataArray that has the same coordinates and metadata
    evi = red.copy()
    # Calculate the EVI
    evi_data = 2.5 * ((nir.data - red.data) / (nir.data + 6.0 * red.data - 7.5 * blue.data + 1.0))
    # Replace the Red xarray.DataArray data with the new EVI data
    evi.data = evi_data
    # exclude the inf values
    evi = xr.where(evi != np.inf, evi, np.nan, keep_attrs=True)
    # change the long_name in the attributes
    evi.attrs['long_name'] = 'EVI'
    evi.attrs['scale_factor'] = 1
    return evi

# make the EVI band
evi = calc_evi(red_sc, blue_sc, nir_sc) # Generate EVI array
evi.hvplot.image(
    aspect='equal',
    cmap='YlGn',
    frame_height=500,
    frame_width= 500
).opts(
    title=f'HLS-derived EVI, {evi.SENSING_TIME}',
    clabel='EVI')

In [ ]:
# save the EVI image out
out_fp = os.path.join(maindir, 'data/imagery/HLS/CravenCanyon_EVI_Example.tif')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)

evi.rio.to_raster(
    out_fp,
    driver="GTiff",           # Specify GeoTIFF driver (default for .tif)
    compress="zstd",          # Use ZSTD compression (efficient and lossless)
    zstd_level=1,             # Compression level
    tiled=True,               # Enable internal tiling
    predictor=2,              # Use predictor for better compression results
    num_threads="all_cpus"    # Utilize all available CPU threads for writing
)
print(f"Saved to: {out_fp}")